In [1]:
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "src")))


In [2]:
# --- Flexible automatic path detection ---
POSSIBLE_PATHS = [
    os.path.join(os.getcwd(), "data", "raw"),                     # notebooks/data/raw
    os.path.join(os.getcwd(), "..", "data", "raw"),               # project_root/data/raw
    os.path.join(os.path.abspath(os.path.join(os.getcwd(), "..")), "data", "raw")  # fallback absolute
]

RAW_DIR = next((p for p in POSSIBLE_PATHS if os.path.exists(p)), None)

if not RAW_DIR:
    raise FileNotFoundError("❌ Could not locate data/raw folder — please verify its path.")

print(f"📁 Using scorer data from: {RAW_DIR}")


📁 Using scorer data from: d:\Codings\AJi FInal\notebooks\..\data\raw


In [3]:
txt_files = [f for f in os.listdir(RAW_DIR) if f.lower().endswith(".txt")]
print(f"🧩 Found {len(txt_files)} scorer files:", txt_files)


🧩 Found 4 scorer files: ['brux1.txt', 'ins1.txt', 'n1.txt', 'sdb1.txt']


In [4]:
import os
import re
import pandas as pd
import numpy as np

def parse_remlogic_event_file_v2(path: str):
    """
    Robust RemLogic parser that detects the data table automatically.
    Works with full event exports (.txt) containing CAP and sleep stages.
    """
    lines = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        lines = [line.strip() for line in f if line.strip()]

    # Find where event data actually starts
    start_idx = None
    for i, line in enumerate(lines):
        if re.search(r"Event\s+Name", line, re.IGNORECASE) or re.search(r"Event\s+Type", line, re.IGNORECASE):
            start_idx = i + 1
            break

    if start_idx is None:
        print(f"⚠️ No 'Event Name' section found in {path}")
        return pd.DataFrame()

    data_lines = lines[start_idx:]
    events = []

    for line in data_lines:
        # Stop if we reach another header or end section
        if re.match(r"^-{3,}$", line) or "Page" in line or "Report" in line:
            continue

        parts = re.split(r"\t+|\s{2,}", line)
        if len(parts) < 3:
            continue

        event_name = parts[0].strip()
        time_token = next((p for p in parts if re.match(r"^\d{1,2}:\d{2}:\d{2}$", p)), None)
        dur_token = next((p for p in parts if re.match(r"^\d+(\.\d+)?$", p)), None)

        if not time_token or not dur_token:
            continue

        subtype = None
        if "A1" in event_name.upper():
            subtype = 1
        elif "A2" in event_name.upper():
            subtype = 2
        elif "A3" in event_name.upper():
            subtype = 3

        events.append({
            "event": event_name,
            "start_time": time_token,
            "duration_s": float(dur_token),
            "subtype": subtype,
        })

    if not events:
        print(f"⚠️ No CAP or Sleep events found in {path}")
        return pd.DataFrame()

    df = pd.DataFrame(events)

    # Convert HH:MM:SS to seconds
    def hms_to_sec(t):
        h, m, s = [float(x) for x in t.split(":")]
        return h * 3600 + m * 60 + s

    df["start_s"] = df["start_time"].apply(hms_to_sec)
    print(f"✅ Parsed {len(df)} events from {os.path.basename(path)}")
    return df


In [5]:
with open(os.path.join(RAW_DIR, "brux1.txt"), "r", encoding="utf-8", errors="ignore") as f:
    for i, line in enumerate(f):
        if i >= 150 and i <= 200:  # show lines 150–200
            print(i, line.strip())


150 S4	Unknown Position	23.02.47	SLEEP-S4	30	EEG-Fp2-F4
151 S4	Unknown Position	23.03.09	MCAP-A1	6	EEG-Fp2-F4
152 S4	Unknown Position	23.03.17	SLEEP-S4	30	EEG-Fp2-F4
153 S4	Unknown Position	23.03.40	MCAP-A1	7	EEG-Fp2-F4
154 S4	Unknown Position	23.03.47	SLEEP-S4	30	EEG-Fp2-F4
155 S4	Unknown Position	23.03.52	MCAP-A1	9	EEG-Fp2-F4
156 S4	Unknown Position	23.04.08	MCAP-A1	3	EEG-F4-C4
157 S4	Unknown Position	23.04.17	SLEEP-S4	30	EEG-Fp2-F4
158 S4	Unknown Position	23.04.41	MCAP-A1	5	EEG-Fp2-F4
159 S4	Unknown Position	23.04.47	SLEEP-S4	30	EEG-Fp2-F4
160 S4	Unknown Position	23.04.52	MCAP-A1	7	EEG-Fp2-F4
161 S4	Unknown Position	23.05.07	MCAP-A3	27	EEG-Fp2-F4
162 S3	Unknown Position	23.05.17	SLEEP-S3	30	EEG-Fp2-F4
163 S2	Unknown Position	23.05.47	SLEEP-S2	30	EEG-Fp2-F4
164 S2	Unknown Position	23.05.57	MCAP-A1	10	EEG-Fp2-F4
165 S2	Unknown Position	23.06.16	MCAP-A3	17	EEG-Fp2-F4
166 S2	Unknown Position	23.06.17	SLEEP-S2	30	EEG-Fp2-F4
167 S2	Unknown Position	23.06.47	SLEEP-S2	30	EEG-Fp2-F4
168 S2	U

In [ ]:
import os

for f in txt_files:
    path = os.path.join(RAW_DIR, f)
    print(f"\n📄 Preview of {f}:")
    print("-" * 60)
    try:
        with open(path, "r", encoding="utf-8", errors="ignore") as file:
            for i in range(20):  # show first 20 lines
                line = file.readline()
                if not line:
                    break
                print(line.strip())
    except Exception as e:
        print(f"⚠️ Error reading {f}: {e}")



📄 Preview of brux1.txt:
------------------------------------------------------------
RemLogic Event Export
Patient:	BRUX 1
Patient ID:
Recording Date:	01/01/2010

Events Included:
MCAP-A1
MCAP-A2
MCAP-A3
SLEEP-MT
SLEEP-REM
SLEEP-S0
SLEEP-S1
SLEEP-S2
SLEEP-S3
SLEEP-S4
SLEEP-UNSCORED

Scoring Session:
Scorer Name:

📄 Preview of ins1.txt:
------------------------------------------------------------
RemLogic Event Export
Patient:	INS1
Patient ID:
Recording Date:	01/01/2006

Events Included:
MCAP-A1
MCAP-A2
MCAP-A3
SLEEP-REM
SLEEP-S0
SLEEP-S1
SLEEP-S2
SLEEP-S3
SLEEP-S4
SLEEP-UNSCORED

Scoring Session:
Scorer Name:
Scoring Time:

📄 Preview of n1.txt:
------------------------------------------------------------
RemLogic Event Export
Patient:	N 1
Patient ID:
Recording Date:	01/01/2009

Events Included:
MCAP-A1
MCAP-A2
MCAP-A3
SLEEP-REM
SLEEP-S0
SLEEP-S1
SLEEP-S2
SLEEP-S3
SLEEP-S4
SLEEP-UNSCORED

Scoring Session:
Scorer Name:
Scoring Time:

📄 Preview of sdb1.txt:
------------------------------

In [7]:
import os

# ✅ Detect root folder dynamically (works from any notebook location)
ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
RAW_DIR = os.path.join(ROOT_DIR, "data", "raw")

# ✅ Verify
print(f"🔍 Using data directory: {RAW_DIR}")
print("Contents:", os.listdir(RAW_DIR))


🔍 Using data directory: d:\Codings\AJi FInal\data\raw
Contents: ['ANNOTATORS', 'brux1.edf', 'brux1.edf .st', 'brux1.txt', 'gender-age.xlsx', 'ins1.edf', 'ins1.edf.st', 'ins1.txt', 'n1.edf', 'n1.edf.st', 'n1.txt', 'RECORDS', 'sdb1.edf', 'sdb1.edf.st', 'sdb1.txt']


In [8]:
pip install os

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement os (from versions: none)
ERROR: No matching distribution found for os


In [9]:
pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [10]:
from phase1_preprocessing.scoring_reader import read_remlogic_txt

for f in ["brux1.txt", "ins1.txt", "n1.txt", "sdb1.txt"]:
    path = os.path.join("data/raw", f)
    print(f"\n🔍 Parsing {f}")
    hyp, ttot, dur, typ = read_remlogic_txt(path)
    print(f"  Hyp rows: {len(hyp)} | CAP events: {len(ttot)}")
    if ttot:
        print("  Sample CAP events:", list(zip(ttot[:5], dur[:5], typ[:5])))


ModuleNotFoundError: No module named 'phase1_preprocessing'